# Action Item Extractor — BIO Token Classification

Token-level sequence labelling that pulls the **fields** out of an action item, rather than
scoring the sentence as a whole:

```
The   industrial  designer     will  prepare   the       working   design    before      the         next        meeting   .
O     B-ASSIGNEE  I-ASSIGNEE   O     B-ACTION  I-ACTION  I-ACTION  I-ACTION  B-DEADLINE  I-DEADLINE  I-DEADLINE  I-DEADLINE O
```

Decoded output: `{"action": "prepare the working design", "assignee": "industrial designer", "deadline": "before the next meeting"}`

**Tag set (7 labels):** `O`, `B-ACTION`, `I-ACTION`, `B-ASSIGNEE`, `I-ASSIGNEE`, `B-DEADLINE`, `I-DEADLINE`

---

## Why this notebook is structured differently

The importance and decision notebooks are **sentence-level binary classifiers** — one label per
sentence, evaluated with F1 / AUC-ROC / PR-AUC. This is a **token-level tagger**: one label per
word, evaluated with **entity-level F1** (a span counts as correct only if its boundaries *and*
type match exactly). ROC and PR curves do not apply, so those sections are replaced by
per-entity metric tables and span-level error analysis.

## Training data problem, and how it is solved

AMI ships **abstractive action sentences but no token-level tags**, and
`data/labeled/action_labels.json` holds only a handful of hand-tagged examples. So the labels are
built in three layers:

| Layer | Source | Quality | Used for |
|---|---|---|---|
| **Silver** | AMI action sentences auto-tagged by a rule tagger | weak supervision | train + test |
| **Augmented** | Silver sentences with a synthetic deadline appended | synthetic | **train only** |
| **Gold** | `data/labeled/action_labels.json`, hand-tagged | human | **test only** |

Augmented examples never enter the test set and gold examples never enter the training set, so
reported scores stay honest.

## Pipeline
1. Parse AMI, auto-tag with the rule tagger, add all-`O` negatives
2. Explore tag and span statistics
3. Split, then augment the training split only
4. Baselines: the rule tagger itself, plus per-token Logistic Regression / Random Forest
5. Fine-tune `bert-base-uncased` for token classification
6. Compare with entity-level F1, run error analysis, save the model and an `extract_actions()` helper

## 1. Setup

In [ ]:
import json
import re
import pickle
import random
import subprocess
import sys
import warnings
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

# ── Colab bootstrap ────────────────────────────────────────────────────────────
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print('Colab detected — installing seqeval (torch/transformers ship with Colab)...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'seqeval'], check=False)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
PALETTE = ['#4C72B0', '#DD8452', '#55A868', '#C44E52',
           '#8172B2', '#937860', '#DA8BC3', '#8C8C8C']


def find_project_root() -> Path:
    """Find the folder that holds amicorpus/, wherever the notebook is running.

    Covers running from notebooks/ locally, from the project root, and the usual
    Colab layouts (/content after unzipping, or Google Drive).
    """
    candidates = [
        Path('..'), Path('.'),
        Path('/content'),
        Path('/content/project'),
        Path('/content/graduation-onsteroids'),
        Path('/content/drive/MyDrive/Graduation Project'),
        Path('/content/drive/MyDrive/graduation-project'),
    ]
    for cand in candidates:
        if (cand / 'amicorpus' / 'abstractive').is_dir():
            return cand
    raise FileNotFoundError(
        'Could not find amicorpus/abstractive/.\n'
        'Upload the project first, then set ROOT manually, e.g.:\n'
        "    ROOT = Path('/content/your_folder')"
    )


ROOT        = find_project_root()
AMI_ROOT    = ROOT / 'amicorpus'
LABELED_DIR = ROOT / 'data' / 'labeled'
OUT_DIR     = ROOT / 'data' / 'processed'
MODEL_DIR   = ROOT / 'models'
OUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# ── Tag set ────────────────────────────────────────────────────────────────────
LABEL_LIST = ['O',
              'B-ACTION',   'I-ACTION',
              'B-ASSIGNEE', 'I-ASSIGNEE',
              'B-DEADLINE', 'I-DEADLINE']
LABEL2ID   = {l: i for i, l in enumerate(LABEL_LIST)}
ID2LABEL   = {i: l for l, i in LABEL2ID.items()}
ENTITY_TYPES = ['ACTION', 'ASSIGNEE', 'DEADLINE']

ENTITY_COLORS = {'ACTION': PALETTE[0], 'ASSIGNEE': PALETTE[1], 'DEADLINE': PALETTE[2]}

print(f'AMI corpus : {AMI_ROOT}')
print(f'Output dir : {OUT_DIR}')
print(f'Model dir  : {MODEL_DIR}')
print(f'Tag set ({len(LABEL_LIST)}): {LABEL_LIST}')

## 2. Build the BIO Dataset

### 2.1 Parse the AMI abstractive summaries

`<actions>` sentences become the positive pool; `<abstract>`, `<decisions>` and `<problems>`
sentences become the all-`O` negative pool so the tagger learns when to predict nothing.

In [ ]:
def parse_abssumm(xml_path: Path) -> dict:
    """Parse one .abssumm.xml file into its four sections."""
    result = {'abstract': [], 'actions': [], 'decisions': [], 'problems': []}
    try:
        root = ET.parse(xml_path).getroot()
    except ET.ParseError:
        return result
    for tag in result:
        section = root.find(tag)
        if section is None:
            continue
        for sent in section.findall('sentence'):
            text = (sent.text or '').strip()
            # AMI uses *NA* / NA. as placeholders for empty sections
            if text and re.sub(r'[^A-Za-z]', '', text).upper() != 'NA':
                result[tag].append(text)
    return result


xml_files = sorted((AMI_ROOT / 'abstractive').glob('*.abssumm.xml'))
parsed    = [parse_abssumm(f) for f in xml_files]

action_sents = [s for m in parsed for s in m['actions']]
other_sents  = [s for m in parsed for k in ['abstract', 'decisions', 'problems'] for s in m[k]]

print(f'Meeting files parsed : {len(xml_files)}')
print(f'Action sentences     : {len(action_sents)}')
print(f'Non-action sentences : {len(other_sents)}')
print()
print('Examples:')
for s in action_sents[:5]:
    print(f'  + {s}')

### 2.2 Rule-based auto-tagger (weak supervision)

AMI action sentences follow a highly regular template — *"The `<role>` will `<verb phrase>`
(`<deadline>`)"* — which makes them tractable for a rule tagger:

1. **Trigger** — locate the first obligation cue (`will`, `is going to`, `has to`, `was instructed to`, …)
2. **ASSIGNEE** — the subject before the trigger, but only when it names a person: a role
   (*"The industrial designer"*) or a bare capitalised name (*"Ahmed"*). *"The prototype will have
   to be redesigned"* has no human assignee, so it stays `O`, and so do pronoun subjects like *"We"*
3. **DEADLINE** — a temporal phrase, accepted only when it is sentence-final, so that a
   mid-sentence temporal (*"thirty minutes after lunch to work on X"*) does not truncate the action.
   A leading preposition stays outside the span (`by`/`O` + `Friday`/`B-DEADLINE`), matching the
   convention already used in `data/labeled/action_labels.json`
4. **ACTION** — everything from just after the trigger up to the deadline or the end

Sentences with two clauses (*"A will X, while B will Y"*) are split so each clause becomes its own example.

In [ ]:
TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:['\-][A-Za-z0-9]+)*|[^\sA-Za-z0-9]")

# Obligation cues, longest first so multi-word triggers win over bare "will"
TRIGGERS = [
    ['was', 'instructed', 'to'], ['were', 'instructed', 'to'],
    ['was', 'asked', 'to'],      ['were', 'asked', 'to'],
    ['is', 'going', 'to'],       ['are', 'going', 'to'],
    ['is', 'to'],  ['are', 'to'],
    ['has', 'to'], ['have', 'to'], ['had', 'to'],
    ['needs', 'to'], ['need', 'to'],
    ['will', 'be'], ['will'], ['shall'], ['should'], ['must'], ['would'],
    ['agreed', 'to'], ['volunteered', 'to'],
]

MONTHS = r'(?:january|february|march|april|may|june|july|august|september|october|november|december)'
DEADLINE_PATTERNS = [
    r'before the next meeting', r'for the next meeting', r'at the next meeting',
    r'during the next meeting', r'in the next meeting', r'by the next meeting',
    r'after the meeting', r'before the meeting', r'next meeting', r'next time',
    r'next week', r'this week', r'tomorrow', r'tonight', r'today',
    r'by the end of the day', r'by the end', r'as soon as possible', r'asap',
    r'after lunch', r'this afternoon', r'this morning', r'in the morning',
    r'by then', r'before they finish',
    rf'on {MONTHS} \d+(?:st|nd|rd|th)?', rf'in {MONTHS}',
    r'by (?:monday|tuesday|wednesday|thursday|friday|saturday|sunday)',
    r'on (?:monday|tuesday|wednesday|thursday|friday|saturday|sunday)',
]
DEADLINE_RE = re.compile(r'\b(' + '|'.join(DEADLINE_PATTERNS) + r')\b', re.I)

LEADING_DET = {'the', 'a', 'an'}

# A subject only becomes an ASSIGNEE if it looks like a person or a role.
ROLE_HINT_RE = re.compile(
    r'\b(manager|designer|deigner|expert|specialist|marketer|team|group|member|members|'
    r'everyone|everybody|participants|people|staff|pm|me|id|ui|us|they|he|she|'
    r'marketing|industrial|interface|project|user)\b', re.I
)

# Capitalised words that open a sentence without being a person's name.
NON_PERSON_TOKENS = {
    'we', 'i', 'you', 'they', 'he', 'she', 'it', 'this', 'that', 'these', 'those',
    'there', 'each', 'every', 'some', 'all', 'both', 'most', 'no', 'one', 'two',
    'individual', 'individuals', 'after', 'before', 'during', 'following', 'his',
    'her', 'their', 'our', 'my', 'another', 'other', 'such', 'any',
}

CLAUSE_SPLIT_RE = re.compile(r',\s+while\s+|\s+and then\s+|;\s+', re.I)
PREP_BEFORE_DEADLINE = {'by', 'before', 'for', 'at', 'in', 'during', 'on', 'after'}


def looks_like_person(tokens: list) -> bool:
    """True for a bare capitalised name such as "Ahmed" or "Sara Khan".

    AMI writes roles as "The industrial designer", so a subject with no leading
    determiner that is title-cased is almost always a personal name. Pronouns are
    excluded so that "We need to update the docs" keeps an empty assignee, matching
    the convention used in data/labeled/action_labels.json.
    """
    if not tokens or len(tokens) > 3:
        return False
    words = [t for t in tokens if re.match(r'[A-Za-z]', t)]
    if not words:
        return False
    if any(w.lower() in NON_PERSON_TOKENS for w in words):
        return False
    return all(w[0].isupper() for w in words)


def deadline_tags(phrase_tokens: list) -> list:
    """BIO tags for a temporal phrase, leaving a leading preposition outside the span.

    data/labeled/action_labels.json tags "by Friday" as O + B-DEADLINE, so silver
    labels follow the same convention or exact-match F1 would punish the model for
    a disagreement between annotators rather than a real error.
    """
    tags = []
    started = False
    for i, tok in enumerate(phrase_tokens):
        if i == 0 and tok.lower() in PREP_BEFORE_DEADLINE:
            tags.append('O')
            continue
        tags.append('I-DEADLINE' if started else 'B-DEADLINE')
        started = True
    return tags


def tokenize(text: str) -> list:
    """Word/punctuation tokenizer kept consistent across silver and gold labels."""
    return TOKEN_RE.findall(text)


def find_trigger(tokens: list):
    """Return (start_index, length) of the earliest obligation cue, or None."""
    lower = [t.lower() for t in tokens]
    best = None
    for trig in TRIGGERS:
        n = len(trig)
        for i in range(len(lower) - n + 1):
            if lower[i:i + n] == trig:
                if best is None or i < best[0]:
                    best = (i, n)
                break
    return best


def auto_tag(text: str):
    """Return (tokens, tags) for one clause, or None if no action structure is found."""
    tokens = tokenize(text)
    if len(tokens) < 3:
        return None
    trig = find_trigger(tokens)
    if trig is None:
        return None
    t_start, t_len = trig
    if t_start == 0:
        return None

    tags = ['O'] * len(tokens)

    # ── ASSIGNEE ───────────────────────────────────────────────────────────────
    a_start = 1 if tokens[0].lower() in LEADING_DET else 0
    a_end   = t_start
    subject_tokens = tokens[a_start:a_end]
    subject = ' '.join(subject_tokens)
    is_assignee = bool(ROLE_HINT_RE.search(subject)) or (
        a_start == 0 and looks_like_person(subject_tokens)
    )
    if a_end - a_start >= 1 and is_assignee:
        tags[a_start] = 'B-ASSIGNEE'
        for i in range(a_start + 1, a_end):
            tags[i] = 'I-ASSIGNEE'

    # trim trailing punctuation so spans never end on a full stop
    act_end = len(tokens)
    while act_end > 0 and not re.match(r'[A-Za-z0-9]', tokens[act_end - 1]):
        act_end -= 1

    # ── DEADLINE (sentence-final only) ─────────────────────────────────────────
    lower_tokens = [t.lower() for t in tokens]
    dl_span = None
    for m in DEADLINE_RE.finditer(text.lower()):
        phrase = tokenize(m.group(0).lower())
        for i in range(t_start + t_len, len(tokens) - len(phrase) + 1):
            if lower_tokens[i:i + len(phrase)] == phrase:
                dl_span = (i, i + len(phrase))
                break
        if dl_span:
            break
    if dl_span and dl_span[1] < act_end - 1:
        dl_span = None

    if dl_span:
        d_start, d_end = dl_span
        for offset, tag in enumerate(deadline_tags(tokens[d_start:d_end])):
            tags[d_start + offset] = tag
        act_end = min(act_end, d_start)
        while act_end > 0 and tokens[act_end - 1].lower() in PREP_BEFORE_DEADLINE:
            act_end -= 1

    # ── ACTION ─────────────────────────────────────────────────────────────────
    act_start = t_start + t_len
    while act_start < act_end and not re.match(r'[A-Za-z0-9]', tokens[act_start]):
        act_start += 1
    if act_end - act_start < 1:
        return None
    tags[act_start] = 'B-ACTION'
    for i in range(act_start + 1, act_end):
        tags[i] = 'I-ACTION' if re.match(r'[A-Za-z0-9]', tokens[i]) else 'O'

    return tokens, tags


def split_clauses(text: str) -> list:
    parts = [p.strip() for p in CLAUSE_SPLIT_RE.split(text) if p.strip()]
    return parts if parts else [text]


# ── Quick demo on a handful of sentences ───────────────────────────────────────
for demo in action_sents[:3]:
    out = auto_tag(demo)
    if out:
        toks, tgs = out
        print(' '.join(f'{t}[{g}]' if g != 'O' else t for t, g in zip(toks, tgs)))
        print()

In [ ]:
# ── Run the auto-tagger over every AMI action sentence ─────────────────────────
silver, failed = [], []
for sent in action_sents:
    for clause in split_clauses(sent):
        out = auto_tag(clause)
        if out is None:
            failed.append(clause)
        else:
            toks, tgs = out
            silver.append({'tokens': toks, 'tags': tgs, 'source': 'ami_silver'})

coverage = len(silver) / max(len(silver) + len(failed), 1)
ent_counts = Counter(t[2:] for ex in silver for t in ex['tags'] if t.startswith('B-'))

print(f'Clauses tagged : {len(silver)}')
print(f'Clauses skipped: {len(failed)}')
print(f'Coverage       : {coverage:.1%}')
print(f'Entity counts  : {dict(ent_counts)}')
print()
print('Sentences the rules could not handle:')
for f in failed[:6]:
    print(f'  - {f}')

### 2.3 Negative sentences

Without all-`O` examples the tagger would learn that *every* sentence contains an action.
Non-action sentences are sampled at roughly 1:1 against the positives.

In [ ]:
random.seed(SEED)
n_neg = min(len(silver), len(other_sents))
neg_sample = random.sample(other_sents, n_neg)

negatives = []
for sent in neg_sample:
    toks = tokenize(sent)
    if len(toks) >= 3:
        negatives.append({'tokens': toks, 'tags': ['O'] * len(toks), 'source': 'ami_negative'})

print(f'Negative (all-O) examples: {len(negatives)}')
print()
ex = negatives[0]
print('Example:', ' '.join(ex['tokens'][:20]), '...')
print('Tags   :', ' '.join(ex['tags'][:20]), '...')

### 2.4 Manual gold labels

`data/labeled/action_labels.json` is already in `{tokens, tags}` BIO format. These are the only
human-annotated examples, so they are held out entirely for evaluation.

In [ ]:
def load_gold_bio(path: Path) -> list:
    """Load hand-tagged BIO examples, keeping only known labels."""
    if not path.exists():
        print(f'No gold file at {path}')
        return []
    with open(path, encoding='utf-8') as f:
        items = json.load(f)

    gold = []
    for item in items:
        toks, tags = item.get('tokens'), item.get('tags')
        if not toks or not tags or len(toks) != len(tags):
            continue
        unknown = set(tags) - set(LABEL_LIST)
        if unknown:
            print(f'  skipped example with unknown tags: {unknown}')
            continue
        gold.append({'tokens': list(toks), 'tags': list(tags), 'source': 'manual_gold'})
    return gold


gold = load_gold_bio(LABELED_DIR / 'action_labels.json')
print(f'Gold examples: {len(gold)}')
for ex in gold[:3]:
    print(' ', ' '.join(f'{t}[{g}]' if g != 'O' else t
                        for t, g in zip(ex['tokens'], ex['tags'])))

if len(gold) < 80:
    print()
    print(f'NOTE: only {len(gold)} gold examples — the project plan targets 80.')
    print('      Entity scores on the gold subset will be very noisy until more are annotated.')

### 2.5 Assemble and save

In [ ]:
dataset = silver + negatives + gold
random.seed(SEED)
random.shuffle(dataset)

src_counts = Counter(ex['source'] for ex in dataset)
tag_counts = Counter(t for ex in dataset for t in ex['tags'])

print(f'Total examples: {len(dataset)}')
for k, v in src_counts.items():
    print(f'  {k:<15}: {v}')
print()
print('Tag distribution:')
for lbl in LABEL_LIST:
    print(f'  {lbl:<12}: {tag_counts.get(lbl, 0)}')

bio_path = OUT_DIR / 'action_bio.json'
with open(bio_path, 'w', encoding='utf-8') as f:
    json.dump(dataset, f, indent=2, ensure_ascii=False)
print(f'\nSaved → {bio_path}')

## 3. Data Exploration

In [ ]:
def extract_spans(tags: list) -> list:
    """Decode a BIO tag sequence into (entity_type, start, end) spans."""
    spans, start, etype = [], None, None
    for i, tag in enumerate(tags + ['O']):
        if tag.startswith('B-'):
            if etype is not None:
                spans.append((etype, start, i))
            etype, start = tag[2:], i
        elif tag.startswith('I-') and etype == tag[2:]:
            continue
        else:
            if etype is not None:
                spans.append((etype, start, i))
            etype, start = None, None
    return spans


fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# ── Tag frequency (excluding O, which dominates) ───────────────────────────────
tag_no_o = {l: tag_counts.get(l, 0) for l in LABEL_LIST if l != 'O'}
bars = axes[0].bar(list(tag_no_o.keys()), list(tag_no_o.values()),
                   color=[ENTITY_COLORS[l[2:]] for l in tag_no_o],
                   edgecolor='white', linewidth=1.2)
axes[0].bar_label(bars, fmt='%d', fontsize=9, padding=2)
axes[0].set_title(f'Tag Frequency  (O = {tag_counts.get("O", 0):,} omitted)',
                  fontsize=12, fontweight='bold')
axes[0].set_ylabel('Token count')
axes[0].tick_params(axis='x', rotation=35)

# ── Sentences containing each entity type ──────────────────────────────────────
sent_with = Counter()
for ex in dataset:
    for etype in {s[0] for s in extract_spans(ex['tags'])}:
        sent_with[etype] += 1
bars = axes[1].bar(ENTITY_TYPES, [sent_with.get(e, 0) for e in ENTITY_TYPES],
                   color=[ENTITY_COLORS[e] for e in ENTITY_TYPES],
                   edgecolor='white', linewidth=1.2)
axes[1].bar_label(bars, fmt='%d', fontsize=10, padding=2)
axes[1].set_title('Sentences Containing Each Entity', fontsize=12, fontweight='bold')
axes[1].set_ylabel('Sentence count')

# ── Span length distribution ───────────────────────────────────────────────────
span_lens = defaultdict(list)
for ex in dataset:
    for etype, s, e in extract_spans(ex['tags']):
        span_lens[etype].append(e - s)
for etype in ENTITY_TYPES:
    if span_lens[etype]:
        axes[2].hist(span_lens[etype], bins=range(1, 20), alpha=0.6,
                     color=ENTITY_COLORS[etype], label=etype, edgecolor='white')
axes[2].set_title('Span Length Distribution', fontsize=12, fontweight='bold')
axes[2].set_xlabel('Tokens per span')
axes[2].set_ylabel('Frequency')
axes[2].legend()

plt.tight_layout()
plt.savefig(MODEL_DIR / 'action_ext_data_overview.png', dpi=150, bbox_inches='tight')
plt.show()

for etype in ENTITY_TYPES:
    lens = span_lens[etype]
    if lens:
        print(f'{etype:<9}: {len(lens):4d} spans   mean length {np.mean(lens):.1f} tokens   max {max(lens)}')

In [ ]:
# ── Class imbalance inside the tag set ────────────────────────────────────────
total_tokens = sum(tag_counts.values())
o_share      = tag_counts.get('O', 0) / total_tokens

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))

axes[0].pie([tag_counts.get('O', 0), total_tokens - tag_counts.get('O', 0)],
            labels=[f'O ({o_share:.1%})', f'Entity tokens ({1 - o_share:.1%})'],
            colors=[PALETTE[7], PALETTE[0]], autopct='%1.1f%%',
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Token-Level Class Balance', fontsize=12, fontweight='bold')

sent_lens = [len(ex['tokens']) for ex in dataset]
axes[1].hist(sent_lens, bins=40, color=PALETTE[0], edgecolor='white')
axes[1].axvline(np.mean(sent_lens), color=PALETTE[3], linestyle='--', linewidth=1.8,
                label=f'mean = {np.mean(sent_lens):.1f}')
axes[1].axvline(np.percentile(sent_lens, 99), color=PALETTE[1], linestyle=':', linewidth=1.8,
                label=f'99th pct = {np.percentile(sent_lens, 99):.0f}')
axes[1].set_title('Sentence Length (tokens)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Tokens')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.savefig(MODEL_DIR / 'action_ext_balance.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Token-level imbalance : {o_share:.1%} of tokens are O')
print(f'Max sentence length   : {max(sent_lens)} tokens  '
      f'(BERT max_len=64 covers {np.mean([l <= 64 for l in sent_lens]):.1%} of sentences)')

## 4. Train / Test Split

Splitting rules that keep the evaluation honest:

- **Gold** (hand-annotated) examples go entirely to **test** — they are the only trustworthy labels
- **Silver** and **negative** examples are split 80/20
- Augmentation happens **after** the split and touches **train only**

In [ ]:
from sklearn.model_selection import train_test_split

silver_and_neg = [ex for ex in dataset if ex['source'] != 'manual_gold']
gold_examples  = [ex for ex in dataset if ex['source'] == 'manual_gold']

# Stratify on "does this sentence contain any entity" to keep positives balanced
strat = [int(any(t != 'O' for t in ex['tags'])) for ex in silver_and_neg]

train_data, test_silver = train_test_split(
    silver_and_neg, test_size=0.20, random_state=SEED, stratify=strat
)
test_data = test_silver + gold_examples

print(f'Train          : {len(train_data)} examples')
print(f'Test (silver)  : {len(test_silver)} examples')
print(f'Test (gold)    : {len(gold_examples)} examples')
print(f'Test total     : {len(test_data)} examples')

for split_name, split in [('train', train_data), ('test', test_data)]:
    c = Counter(t[2:] for ex in split for t in ex['tags'] if t.startswith('B-'))
    print(f'  {split_name:<6} entities: ' + '  '.join(f'{e}={c.get(e, 0)}' for e in ENTITY_TYPES))

### 4.1 Deadline augmentation (training split only)

AMI meeting summaries almost never state an explicit deadline — the auto-tagger found only a
handful in the whole corpus. With so few examples the model cannot learn `DEADLINE` at all.

Each selected **training** sentence that has no deadline gets a realistic temporal phrase appended
and tagged. The test split is left untouched, so this cannot inflate the reported scores.

> This is a stopgap for a data shortage, not a substitute for real annotation. Deadlines in actual
> Telegram messages look different (*"by fri"*, *"eod"*, *"before standup"*), which is exactly what
> the manual labelling effort in `data/labeled/` is for.

In [ ]:
DEADLINE_PHRASES = [
    'by Friday', 'by Monday', 'before the next meeting', 'by tomorrow',
    'next week', 'by the end of the day', 'this afternoon', 'before lunch',
    'by the end of the week', 'tomorrow morning', 'as soon as possible',
    'before the deadline', 'by Wednesday', 'at the next meeting',
]

AUGMENT_FRACTION = 0.35  # share of eligible training sentences to augment


def has_entity(ex, etype):
    return any(t.endswith(etype) for t in ex['tags'])


def augment_with_deadline(ex, phrase):
    """Append a temporal phrase to a sentence and tag it as DEADLINE."""
    tokens = list(ex['tokens'])
    tags   = list(ex['tags'])

    # insert before any trailing punctuation so the sentence stays well-formed
    tail = []
    while tokens and not re.match(r'[A-Za-z0-9]', tokens[-1]):
        tail.insert(0, tokens.pop())
        tags.pop()

    phrase_tokens = tokenize(phrase)
    tokens += phrase_tokens
    tags   += deadline_tags(phrase_tokens)   # same preposition convention as the gold labels
    tokens += tail
    tags   += ['O'] * len(tail)
    return {'tokens': tokens, 'tags': tags, 'source': 'ami_augmented'}


random.seed(SEED)
eligible = [ex for ex in train_data
            if has_entity(ex, 'ACTION') and not has_entity(ex, 'DEADLINE')]
n_aug    = int(len(eligible) * AUGMENT_FRACTION)
to_augment = random.sample(eligible, n_aug)

augmented = [augment_with_deadline(ex, random.choice(DEADLINE_PHRASES)) for ex in to_augment]
train_data = train_data + augmented

train_dl = sum(1 for ex in train_data if has_entity(ex, 'DEADLINE'))
test_dl  = sum(1 for ex in test_data  if has_entity(ex, 'DEADLINE'))

print(f'Eligible training sentences : {len(eligible)}')
print(f'Augmented                   : {len(augmented)}')
print(f'Train size after augmentation: {len(train_data)}')
print()
print(f'Sentences with a DEADLINE — train: {train_dl}   test: {test_dl}')
print()
ex = augmented[0]
print('Example augmented sentence:')
print(' ', ' '.join(f'{t}[{g}]' if g != 'O' else t for t, g in zip(ex['tokens'], ex['tags'])))

if test_dl < 10:
    print()
    print(f'WARNING: only {test_dl} test sentences contain a DEADLINE.')
    print('         Treat the DEADLINE column of every results table as indicative, not reliable.')

## 5. Evaluation Metric — Entity-Level F1

A span counts as correct only when its **type and both boundaries** match exactly. Predicting
`prepare the working` when the gold span is `prepare the working design` scores zero, not partial
credit — which is why entity F1 always sits below token accuracy.

`seqeval` is the standard implementation; an equivalent fallback is defined so the notebook runs
without it.

In [ ]:
try:
    from seqeval.metrics import classification_report as seq_report
    from seqeval.metrics import f1_score as seq_f1
    HAS_SEQEVAL = True
    print('seqeval available')
except ImportError:
    HAS_SEQEVAL = False
    print('seqeval not installed (pip install seqeval) — using the built-in fallback')


def entity_scores(y_true: list, y_pred: list) -> dict:
    """Entity-level precision/recall/F1, micro-averaged and per entity type."""
    tp = Counter()
    fp = Counter()
    fn = Counter()

    for true_tags, pred_tags in zip(y_true, y_pred):
        true_spans = set(extract_spans(true_tags))
        pred_spans = set(extract_spans(pred_tags))
        for span in pred_spans & true_spans:
            tp[span[0]] += 1
        for span in pred_spans - true_spans:
            fp[span[0]] += 1
        for span in true_spans - pred_spans:
            fn[span[0]] += 1

    def prf(t, f_pos, f_neg):
        p = t / (t + f_pos) if (t + f_pos) else 0.0
        r = t / (t + f_neg) if (t + f_neg) else 0.0
        f = 2 * p * r / (p + r) if (p + r) else 0.0
        return p, r, f

    out = {}
    for etype in ENTITY_TYPES:
        p, r, f = prf(tp[etype], fp[etype], fn[etype])
        out[etype] = {'precision': p, 'recall': r, 'f1': f, 'support': tp[etype] + fn[etype]}

    p, r, f = prf(sum(tp.values()), sum(fp.values()), sum(fn.values()))
    out['micro'] = {'precision': p, 'recall': r, 'f1': f,
                    'support': sum(tp.values()) + sum(fn.values())}
    return out


def token_accuracy(y_true: list, y_pred: list) -> float:
    correct = total = 0
    for t, p in zip(y_true, y_pred):
        for a, b in zip(t, p):
            correct += int(a == b)
            total   += 1
    return correct / max(total, 1)


# ── Sanity check against seqeval ──────────────────────────────────────────────
_t = [['O', 'B-ACTION', 'I-ACTION', 'O'], ['B-ASSIGNEE', 'O', 'B-ACTION', 'O']]
_p = [['O', 'B-ACTION', 'I-ACTION', 'O'], ['B-ASSIGNEE', 'O', 'O',        'O']]
print(f'fallback micro F1 : {entity_scores(_t, _p)["micro"]["f1"]:.4f}')
if HAS_SEQEVAL:
    print(f'seqeval  micro F1 : {seq_f1(_t, _p):.4f}')

y_test_tags = [ex['tags']   for ex in test_data]
X_test_toks = [ex['tokens'] for ex in test_data]

# Every model registers its predictions here for the comparison section
results = {}

## 6. Baseline 1 — The Rule Tagger Itself

The rules generated the training labels, so they should score near-perfectly on silver test
sentences. The number worth reading is the score on the **gold** subset, which shows how well
hand-written rules transfer to human-annotated text.

In [ ]:
def rule_predict(tokens: list) -> list:
    """Run the auto-tagger over pre-tokenized input and return a tag sequence."""
    out = auto_tag(' '.join(tokens))
    if out is None:
        return ['O'] * len(tokens)
    pred_tokens, pred_tags = out
    if len(pred_tags) == len(tokens):
        return pred_tags
    # detokenize/retokenize changed the length — align by padding or truncating
    if len(pred_tags) < len(tokens):
        return pred_tags + ['O'] * (len(tokens) - len(pred_tags))
    return pred_tags[:len(tokens)]


rule_preds  = [rule_predict(toks) for toks in X_test_toks]
rule_scores = entity_scores(y_test_tags, rule_preds)

results['Rule-based'] = {
    'preds'    : rule_preds,
    'scores'   : rule_scores,
    'token_acc': token_accuracy(y_test_tags, rule_preds),
}

print('Rule-based tagger — full test set')
print(f'  micro F1       : {rule_scores["micro"]["f1"]:.4f}')
print(f'  token accuracy : {token_accuracy(y_test_tags, rule_preds):.4f}')
for etype in ENTITY_TYPES:
    s = rule_scores[etype]
    print(f'  {etype:<9}: P={s["precision"]:.4f}  R={s["recall"]:.4f}  F1={s["f1"]:.4f}  (support={s["support"]})')

# ── Gold subset only ──────────────────────────────────────────────────────────
gold_idx = [i for i, ex in enumerate(test_data) if ex['source'] == 'manual_gold']
if gold_idx:
    g_true = [y_test_tags[i] for i in gold_idx]
    g_pred = [rule_preds[i]  for i in gold_idx]
    print()
    print(f'Gold subset only ({len(gold_idx)} hand-annotated sentences)')
    print(f'  micro F1 : {entity_scores(g_true, g_pred)["micro"]["f1"]:.4f}')

## 7. Baseline 2 — Per-Token Classical ML

Each token is classified independently from a window of surrounding features (word shape,
suffixes, neighbouring words, membership in modal / role / temporal lists). No sequence modelling,
so nothing prevents an invalid sequence such as `I-ACTION` following `O` — the gap between these
baselines and BERT is largely the cost of ignoring context.

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

MODAL_SET    = {'will', 'shall', 'should', 'must', 'would', 'going', 'need', 'needs', 'have', 'has'}
TEMPORAL_SET = {'tomorrow', 'today', 'tonight', 'monday', 'tuesday', 'wednesday', 'thursday',
                'friday', 'saturday', 'sunday', 'week', 'meeting', 'morning', 'afternoon',
                'lunch', 'deadline', 'soon', 'next', 'by', 'before'}


def token_features(tokens: list, i: int) -> dict:
    w = tokens[i]
    feats = {
        'bias'        : 1.0,
        'w.lower'     : w.lower(),
        'w[-3:]'      : w[-3:],
        'w[-2:]'      : w[-2:],
        'w.istitle'   : w.istitle(),
        'w.isupper'   : w.isupper(),
        'w.isdigit'   : w.isdigit(),
        'w.ispunct'   : not bool(re.match(r'[A-Za-z0-9]', w)),
        'w.is_modal'  : w.lower() in MODAL_SET,
        'w.is_role'   : bool(ROLE_HINT_RE.fullmatch(w)),
        'w.is_temporal': w.lower() in TEMPORAL_SET,
        'rel_pos'     : i / max(len(tokens), 1),
        'is_first'    : i == 0,
        'is_last'     : i == len(tokens) - 1,
    }
    for offset in (-2, -1, 1, 2):
        j = i + offset
        key = f'{offset:+d}'
        if 0 <= j < len(tokens):
            feats[f'w[{key}].lower']   = tokens[j].lower()
            feats[f'w[{key}].is_modal'] = tokens[j].lower() in MODAL_SET
        else:
            feats[f'w[{key}].pad'] = True
    return feats


def build_token_matrix(examples: list):
    feats, labels, lengths = [], [], []
    for ex in examples:
        toks = ex['tokens']
        for i in range(len(toks)):
            feats.append(token_features(toks, i))
            labels.append(ex['tags'][i])
        lengths.append(len(toks))
    return feats, labels, lengths


train_feats, train_labels, _            = build_token_matrix(train_data)
test_feats,  test_labels,  test_lengths = build_token_matrix(test_data)

vectorizer = DictVectorizer(sparse=True)
X_tr = vectorizer.fit_transform(train_feats)
X_te = vectorizer.transform(test_feats)

print(f'Train tokens : {X_tr.shape[0]:,}   features: {X_tr.shape[1]:,}')
print(f'Test tokens  : {X_te.shape[0]:,}')


def regroup(flat_preds: list, lengths: list) -> list:
    """Turn a flat token prediction list back into per-sentence sequences."""
    out, pos = [], 0
    for n in lengths:
        out.append(list(flat_preds[pos:pos + n]))
        pos += n
    return out


CLASSICAL_MODELS = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced',
                                              random_state=SEED, n_jobs=-1),
    'Random Forest'      : RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                                                  class_weight='balanced', random_state=SEED,
                                                  n_jobs=-1),
}

for name, model in CLASSICAL_MODELS.items():
    print(f'Training {name}...', end=' ')
    model.fit(X_tr, train_labels)
    preds  = regroup(model.predict(X_te).tolist(), test_lengths)
    scores = entity_scores(y_test_tags, preds)
    results[name] = {
        'preds'    : preds,
        'scores'   : scores,
        'token_acc': token_accuracy(y_test_tags, preds),
    }
    print(f'entity F1={scores["micro"]["f1"]:.4f}  token acc={results[name]["token_acc"]:.4f}')

## 8. BERT Token Classification

`bert-base-uncased` with a `Linear(768 → 7)` head on top of every token's final hidden state.

| Step | Detail |
|---|---|
| Model | `bert-base-uncased` + token classification head |
| Max tokens | 64 word-pieces |
| Epochs | 8 (small dataset), best checkpoint kept on entity F1 |
| Batch size | 16 train / 32 eval |
| Learning rate | 3e-5, linear warm-up (10 %) |
| Subword handling | Only the **first** word-piece of each word carries the label; the rest get `-100` and are ignored by the loss |

In [ ]:
try:
    import torch
    from torch.utils.data import Dataset as TorchDataset
    from transformers import (
        AutoTokenizer, AutoModelForTokenClassification,
        TrainingArguments, Trainer, DataCollatorForTokenClassification,
    )
    HAS_TRANSFORMERS = True
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    import transformers as _tr
    print(f'transformers {_tr.__version__}  |  torch {torch.__version__}  |  device: {device}')
except ImportError as e:
    HAS_TRANSFORMERS = False
    print(f'Missing package: {e}')
    print('Run:  pip install transformers torch')
    print('The BERT section will be skipped.')

In [ ]:
if HAS_TRANSFORMERS:
    BERT_MODEL_NAME = 'bert-base-uncased'
    MAX_LEN         = 64
    bert_tokenizer  = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)

    def encode_example(tokens, tags, tokenizer, max_len=MAX_LEN):
        """Tokenize pre-split words and align labels to the first sub-word of each word."""
        enc = tokenizer(
            tokens,
            is_split_into_words=True,
            truncation=True,
            padding='max_length',
            max_length=max_len,
        )
        word_ids = enc.word_ids()
        labels, previous = [], None
        for wid in word_ids:
            if wid is None:
                labels.append(-100)                 # special token
            elif wid != previous:
                labels.append(LABEL2ID[tags[wid]])  # first sub-word carries the label
            else:
                labels.append(-100)                 # continuation sub-word
            previous = wid
        enc['labels'] = labels
        return enc

    class BIODataset(TorchDataset):
        def __init__(self, examples, tokenizer):
            self.encodings = [encode_example(ex['tokens'], ex['tags'], tokenizer)
                              for ex in examples]

        def __len__(self):
            return len(self.encodings)

        def __getitem__(self, idx):
            return {k: torch.tensor(v) for k, v in self.encodings[idx].items()}

    bert_train_ds = BIODataset(train_data, bert_tokenizer)
    bert_test_ds  = BIODataset(test_data,  bert_tokenizer)

    print(f'Train : {len(bert_train_ds)} sentences')
    print(f'Test  : {len(bert_test_ds)} sentences')
    print()

    # Show the alignment on one example
    demo_ex  = train_data[0]
    demo_enc = encode_example(demo_ex['tokens'], demo_ex['tags'], bert_tokenizer)
    pieces   = bert_tokenizer.convert_ids_to_tokens(demo_enc['input_ids'])
    print('Sub-word alignment (first 18 positions):')
    for piece, lab in list(zip(pieces, demo_enc['labels']))[:18]:
        shown = ID2LABEL[lab] if lab != -100 else '(ignored)'
        print(f'  {piece:<15} → {shown}')

In [ ]:
if HAS_TRANSFORMERS:
    def compute_bert_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=2)

        true_seqs, pred_seqs = [], []
        for pred_row, label_row in zip(preds, labels):
            t_seq, p_seq = [], []
            for p_i, l_i in zip(pred_row, label_row):
                if l_i != -100:
                    t_seq.append(ID2LABEL[int(l_i)])
                    p_seq.append(ID2LABEL[int(p_i)])
            true_seqs.append(t_seq)
            pred_seqs.append(p_seq)

        s = entity_scores(true_seqs, pred_seqs)
        return {
            'f1'           : s['micro']['f1'],
            'precision'    : s['micro']['precision'],
            'recall'       : s['micro']['recall'],
            'f1_action'    : s['ACTION']['f1'],
            'f1_assignee'  : s['ASSIGNEE']['f1'],
            'f1_deadline'  : s['DEADLINE']['f1'],
        }

    bert_model = AutoModelForTokenClassification.from_pretrained(
        BERT_MODEL_NAME,
        num_labels=len(LABEL_LIST),
        id2label=ID2LABEL,
        label2id=LABEL2ID,
    )

    BERT_OUTPUT = str(MODEL_DIR / 'bert_action_extractor')

    # evaluation_strategy was renamed to eval_strategy in transformers >= 4.41
    import inspect
    _ta_params = inspect.signature(TrainingArguments.__init__).parameters
    _eval_kw   = 'eval_strategy' if 'eval_strategy' in _ta_params else 'evaluation_strategy'

    training_args = TrainingArguments(
        output_dir                  = BERT_OUTPUT,
        num_train_epochs            = 8,
        per_device_train_batch_size = 16,
        per_device_eval_batch_size  = 32,
        learning_rate               = 3e-5,
        weight_decay                = 0.01,
        warmup_ratio                = 0.1,
        save_strategy               = 'epoch',
        save_total_limit            = 1,
        load_best_model_at_end      = True,
        metric_for_best_model       = 'f1',
        greater_is_better           = True,
        logging_steps               = 20,
        report_to                   = 'none',
        seed                        = SEED,
        **{_eval_kw: 'epoch'},
    )

    bert_trainer = Trainer(
        model           = bert_model,
        args            = training_args,
        train_dataset   = bert_train_ds,
        eval_dataset    = bert_test_ds,
        compute_metrics = compute_bert_metrics,
        data_collator   = DataCollatorForTokenClassification(bert_tokenizer),
    )

    print('Starting BERT fine-tuning...')
    print(f'  Model      : {BERT_MODEL_NAME}')
    print(f'  Labels     : {len(LABEL_LIST)}')
    print(f'  Epochs     : {training_args.num_train_epochs}')
    print(f'  Device     : {device}')
    bert_trainer.train()
    print('\nFine-tuning complete.')

In [ ]:
if HAS_TRANSFORMERS:
    # ── Decode BERT predictions back to word-level tag sequences ──────────────
    bert_out    = bert_trainer.predict(bert_test_ds)
    bert_logits = bert_out.predictions
    bert_labels = bert_out.label_ids
    bert_argmax = np.argmax(bert_logits, axis=2)

    bert_preds, bert_true = [], []
    for pred_row, label_row in zip(bert_argmax, bert_labels):
        p_seq, t_seq = [], []
        for p_i, l_i in zip(pred_row, label_row):
            if l_i != -100:
                p_seq.append(ID2LABEL[int(p_i)])
                t_seq.append(ID2LABEL[int(l_i)])
        bert_preds.append(p_seq)
        bert_true.append(t_seq)

    # Sentences longer than MAX_LEN lose their tail; pad so lengths line up again
    bert_preds_aligned = []
    for pred_seq, ex in zip(bert_preds, test_data):
        n = len(ex['tokens'])
        bert_preds_aligned.append((pred_seq + ['O'] * n)[:n])

    bert_scores = entity_scores(y_test_tags, bert_preds_aligned)
    results['BERT (fine-tuned)'] = {
        'preds'    : bert_preds_aligned,
        'scores'   : bert_scores,
        'token_acc': token_accuracy(y_test_tags, bert_preds_aligned),
    }

    print('BERT token classification — test metrics')
    print(f'  entity F1 (micro): {bert_scores["micro"]["f1"]:.4f}')
    print(f'  precision        : {bert_scores["micro"]["precision"]:.4f}')
    print(f'  recall           : {bert_scores["micro"]["recall"]:.4f}')
    print(f'  token accuracy   : {results["BERT (fine-tuned)"]["token_acc"]:.4f}')
    for etype in ENTITY_TYPES:
        s = bert_scores[etype]
        print(f'  {etype:<9}: P={s["precision"]:.4f}  R={s["recall"]:.4f}  '
              f'F1={s["f1"]:.4f}  (support={s["support"]})')

    if HAS_SEQEVAL:
        print()
        print('seqeval report:')
        print(seq_report(y_test_tags, bert_preds_aligned, digits=4))

    bert_trainer.save_model(BERT_OUTPUT)
    bert_tokenizer.save_pretrained(BERT_OUTPUT)
    print(f'BERT model saved → {BERT_OUTPUT}')
else:
    print('BERT section skipped — the comparison below shows only the baselines.')

## 9. Results & Comparison

In [ ]:
final_df = pd.DataFrame([
    {
        'Model'         : name,
        'Entity F1'     : round(r['scores']['micro']['f1'],        4),
        'Precision'     : round(r['scores']['micro']['precision'], 4),
        'Recall'        : round(r['scores']['micro']['recall'],    4),
        'Token Acc'     : round(r['token_acc'],                    4),
        'F1 ACTION'     : round(r['scores']['ACTION']['f1'],       4),
        'F1 ASSIGNEE'   : round(r['scores']['ASSIGNEE']['f1'],     4),
        'F1 DEADLINE'   : round(r['scores']['DEADLINE']['f1'],     4),
    }
    for name, r in results.items()
]).set_index('Model').sort_values('Entity F1', ascending=False)

# Selected once here so every later section (confusion matrix, error analysis,
# saving) refers to the same model.
best_name  = final_df['Entity F1'].idxmax()
best_preds = results[best_name]['preds']
print(f'Best model: {best_name}  (entity F1 = {final_df.loc[best_name, "Entity F1"]:.4f})')


def _highlight_bert(row):
    if 'BERT' in row.name:
        return ['background-color: #fff3cd; font-weight: bold'] * len(row)
    return [''] * len(row)


display(final_df.style
    .background_gradient(cmap='YlGn',    subset=['Entity F1', 'Precision', 'Recall'])
    .background_gradient(cmap='Oranges', subset=['F1 ACTION', 'F1 ASSIGNEE', 'F1 DEADLINE'])
    .apply(_highlight_bert, axis=1)
    .format('{:.4f}')
    .set_caption('Action Extractor — Entity-Level Comparison (BERT highlighted in gold)')
)

In [ ]:
# ── Overall comparison ────────────────────────────────────────────────────────
names   = list(final_df.index)
is_bert = ['BERT' in n for n in names]

x = np.arange(len(names))
w = 0.26

fig, ax = plt.subplots(figsize=(max(9, len(names) * 2.2), 5))
b1 = ax.bar(x - w, final_df['Entity F1'], w, color=PALETTE[0], alpha=0.88,
            label='Entity F1', edgecolor='white')
b2 = ax.bar(x,     final_df['Precision'], w, color=PALETTE[1], alpha=0.88,
            label='Precision', edgecolor='white')
b3 = ax.bar(x + w, final_df['Recall'],    w, color=PALETTE[2], alpha=0.88,
            label='Recall', edgecolor='white')
for bars in (b1, b2, b3):
    ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=2)
    for bar, flag in zip(bars, is_bert):
        if flag:
            bar.set_edgecolor('goldenrod')
            bar.set_linewidth(2.5)

ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=10, rotation=12, ha='right')
ax.set_ylim(0, 1.18)
ax.set_ylabel('Score')
ax.set_title('Entity-Level Performance — All Models\n(gold outline = BERT fine-tuned)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper right')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'action_ext_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Per-entity F1 ─────────────────────────────────────────────────────────────
x = np.arange(len(ENTITY_TYPES))
w = 0.8 / max(len(results), 1)

fig, ax = plt.subplots(figsize=(10, 5))
for k, (name, r) in enumerate(results.items()):
    vals = [r['scores'][e]['f1'] for e in ENTITY_TYPES]
    bars = ax.bar(x + k * w - 0.4 + w / 2, vals, w,
                  color=PALETTE[k % len(PALETTE)], alpha=0.88,
                  label=name, edgecolor='white')
    ax.bar_label(bars, fmt='%.2f', fontsize=7, padding=2)

supports = [results[list(results)[0]]['scores'][e]['support'] for e in ENTITY_TYPES]
ax.set_xticks(x)
ax.set_xticklabels([f'{e}\n(support={s})' for e, s in zip(ENTITY_TYPES, supports)], fontsize=10)
ax.set_ylim(0, 1.18)
ax.set_ylabel('Entity F1')
ax.set_title('Per-Entity F1 by Model', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(MODEL_DIR / 'action_ext_per_entity.png', dpi=150, bbox_inches='tight')
plt.show()

print('Support = number of gold spans of that type in the test set.')
print('A low support column (typically DEADLINE) makes its F1 unstable.')

In [ ]:
# ── Token-level confusion matrix for the best model ───────────────────────────
from sklearn.metrics import confusion_matrix

flat_true = [t for seq in y_test_tags for t in seq]
flat_pred = [t for seq in best_preds  for t in seq]

cm = confusion_matrix(flat_true, flat_pred, labels=LABEL_LIST)
cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
            xticklabels=LABEL_LIST, yticklabels=LABEL_LIST,
            linewidths=1, linecolor='white', ax=ax,
            cbar_kws={'label': 'Row-normalised share'})
ax.set_xlabel('Predicted', fontsize=11)
ax.set_ylabel('Actual', fontsize=11)
ax.set_title(f'Token-Level Confusion Matrix — {best_name}\n'
             '(cell text = counts, colour = share of the true label)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'action_ext_confusion.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Best model: {best_name}')

## 10. Error Analysis

Entity F1 hides *how* a span was wrong. Every predicted span falls into one of four buckets:

- **exact** — type and boundaries both correct
- **boundary** — right type, wrong start/end (the most common and least harmful failure)
- **type** — right span, wrong label
- **spurious / missed** — predicted where nothing exists, or nothing predicted where it does

In [ ]:
def categorise_errors(true_seqs, pred_seqs):
    """Bucket every predicted and gold span into an error category."""
    buckets = Counter()
    examples = defaultdict(list)

    for idx, (t_tags, p_tags) in enumerate(zip(true_seqs, pred_seqs)):
        t_spans = set(extract_spans(t_tags))
        p_spans = set(extract_spans(p_tags))

        for span in p_spans:
            etype, s, e = span
            if span in t_spans:
                buckets['exact'] += 1
            elif any(ts[0] == etype and not (ts[2] <= s or ts[1] >= e) for ts in t_spans):
                buckets['boundary'] += 1
                examples['boundary'].append(idx)
            elif any(ts[1] == s and ts[2] == e for ts in t_spans):
                buckets['type'] += 1
                examples['type'].append(idx)
            else:
                buckets['spurious'] += 1
                examples['spurious'].append(idx)

        for span in t_spans:
            etype, s, e = span
            if span in p_spans:
                continue
            overlapped = any(ps[0] == etype and not (ps[2] <= s or ps[1] >= e) for ps in p_spans)
            same_bounds = any(ps[1] == s and ps[2] == e for ps in p_spans)
            if not overlapped and not same_bounds:
                buckets['missed'] += 1
                examples['missed'].append(idx)

    return buckets, examples


buckets, err_examples = categorise_errors(y_test_tags, best_preds)
order  = ['exact', 'boundary', 'type', 'spurious', 'missed']
colors = [PALETTE[2], PALETTE[1], PALETTE[4], PALETTE[3], PALETTE[7]]

fig, ax = plt.subplots(figsize=(9, 4.2))
vals = [buckets.get(k, 0) for k in order]
bars = ax.bar(order, vals, color=colors, edgecolor='white', linewidth=1.2)
ax.bar_label(bars, fmt='%d', fontsize=10, padding=3)
ax.set_ylabel('Span count')
ax.set_title(f'Span Error Breakdown — {best_name}', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'action_ext_errors.png', dpi=150, bbox_inches='tight')
plt.show()

total_spans = sum(vals)
for k in order:
    print(f'  {k:<10}: {buckets.get(k, 0):4d}  ({buckets.get(k, 0) / max(total_spans, 1):.1%})')

In [ ]:
# ── Inspect a few mistakes side by side ───────────────────────────────────────
def show_comparison(idx):
    tokens = test_data[idx]['tokens']
    print(f'[{test_data[idx]["source"]}]')
    print('  gold:', ' '.join(f'{t}[{g}]' if g != 'O' else t
                              for t, g in zip(tokens, y_test_tags[idx])))
    print('  pred:', ' '.join(f'{t}[{g}]' if g != 'O' else t
                              for t, g in zip(tokens, best_preds[idx])))
    print()


shown = set()
for category in ['boundary', 'spurious', 'missed', 'type']:
    idxs = [i for i in err_examples.get(category, []) if i not in shown][:2]
    if idxs:
        print('=' * 70)
        print(f'  {category.upper()} errors')
        print('=' * 70)
        for i in idxs:
            shown.add(i)
            show_comparison(i)

if not shown:
    print('No span errors found on the test set.')

## 11. Save & Inference Helper

`extract_actions()` returns the structured output the pipeline actually consumes:
`{action, assignee, deadline}` per detected action item, with an empty list when the sentence
contains none.

In [ ]:
artifact = {
    'label_list'      : LABEL_LIST,
    'label2id'        : LABEL2ID,
    'id2label'        : ID2LABEL,
    'best_model_name' : best_name,
    'bert_path'       : str(MODEL_DIR / 'bert_action_extractor') if HAS_TRANSFORMERS else None,
    'metrics'         : {
        name: {
            'entity_f1' : r['scores']['micro']['f1'],
            'precision' : r['scores']['micro']['precision'],
            'recall'    : r['scores']['micro']['recall'],
            'token_acc' : r['token_acc'],
            'per_entity': {e: r['scores'][e] for e in ENTITY_TYPES},
        }
        for name, r in results.items()
    },
    'dataset_sizes'   : {
        'train': len(train_data),
        'test' : len(test_data),
        'gold' : len(gold_examples),
    },
}

# The classical baseline needs its vectorizer to run again later
if 'Logistic Regression' in CLASSICAL_MODELS:
    artifact['token_vectorizer'] = vectorizer
    artifact['token_model']      = CLASSICAL_MODELS['Logistic Regression']

meta_path = MODEL_DIR / 'action_extractor.pkl'
with open(meta_path, 'wb') as f:
    pickle.dump(artifact, f)

print(f'Artifacts saved → {meta_path}')
if HAS_TRANSFORMERS:
    print(f'BERT weights   → {artifact["bert_path"]}')
print()
print(f'Best model : {best_name}  (entity F1 = {final_df.loc[best_name, "Entity F1"]:.4f})')

In [ ]:
def tags_to_items(tokens: list, tags: list) -> list:
    """Decode a BIO sequence into {action, assignee, deadline} dictionaries."""
    spans = extract_spans(tags)
    if not any(s[0] == 'ACTION' for s in spans):
        return []

    def text_of(span):
        return ' '.join(tokens[span[1]:span[2]])

    # Pair each ACTION with the nearest assignee/deadline in the same sentence
    items = []
    actions   = [s for s in spans if s[0] == 'ACTION']
    assignees = [s for s in spans if s[0] == 'ASSIGNEE']
    deadlines = [s for s in spans if s[0] == 'DEADLINE']

    for act in actions:
        def nearest(candidates):
            if not candidates:
                return None
            best = min(candidates, key=lambda c: abs(c[1] - act[1]))
            return text_of(best)

        items.append({
            'action'  : text_of(act),
            'assignee': nearest(assignees),
            'deadline': nearest(deadlines),
        })
    return items


def extract_actions(sentences, model_path=None):
    """Run the fine-tuned BERT tagger over raw sentences and return structured items."""
    if not HAS_TRANSFORMERS:
        print('transformers unavailable — falling back to the rule tagger.')
        return [{'sentence': s, 'items': tags_to_items(tokenize(s), rule_predict(tokenize(s)))}
                for s in sentences]

    path = model_path or artifact['bert_path']
    tok  = AutoTokenizer.from_pretrained(path)
    mdl  = AutoModelForTokenClassification.from_pretrained(path).to(device).eval()

    output = []
    for sentence in sentences:
        words = tokenize(sentence)
        enc   = tok(words, is_split_into_words=True, truncation=True,
                    max_length=MAX_LEN, return_tensors='pt').to(device)
        with torch.no_grad():
            logits = mdl(**enc).logits[0].cpu().numpy()

        word_ids  = enc.word_ids(0)
        pred_tags = ['O'] * len(words)
        seen = set()
        for pos, wid in enumerate(word_ids):
            if wid is not None and wid not in seen:
                seen.add(wid)
                pred_tags[wid] = ID2LABEL[int(np.argmax(logits[pos]))]

        output.append({'sentence': sentence,
                       'items'   : tags_to_items(words, pred_tags),
                       'tags'    : pred_tags})
    return output


demo_sentences = [
    'The industrial designer will prepare the working design before the next meeting.',
    'Ahmed should fix the login bug by Friday.',
    'Sara will review the pull request tomorrow.',
    'We need to update the API documentation.',
    'We decided to go with the touchscreen interface.',
    'haha yeah that makes sense',
]

for row in extract_actions(demo_sentences):
    print(f'> {row["sentence"]}')
    if row['items']:
        for item in row['items']:
            print(f'    action   : {item["action"]}')
            print(f'    assignee : {item["assignee"]}')
            print(f'    deadline : {item["deadline"]}')
    else:
        print('    (no action item detected)')
    print()